# Historical Scenario Validation — E5.4

This notebook validates the historical scenario infrastructure introduced in
[Epic E5.4](https://github.com/B9android/wargames_training/issues).  
It loads three Napoleonic battle scenarios (Waterloo, Austerlitz, Borodino),
runs the simulation engine, and compares the results against documented
historical outcomes using `OutcomeComparator`.

## Structure

1. **Environment setup** — imports and path configuration  
2. **Scenario loading** — parse YAML, inspect initial conditions  
3. **1v1 baseline runs** — run the raw simulation (no trained policy)  
4. **Outcome comparison** — measure fidelity vs. historical record  
5. **Visualisation** — fidelity bar chart, casualty comparison table  
6. **Summary** — conclusions and next steps

In [ ]:
import sys
from pathlib import Path

# Ensure project root is importable when running from notebooks/
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from envs.scenarios import load_scenario, OutcomeComparator
from envs.sim.engine import SimEngine

SCENARIOS_DIR = PROJECT_ROOT / "configs" / "scenarios" / "historical"
SCENARIO_FILES = {
    "Waterloo (1815)": SCENARIOS_DIR / "waterloo.yaml",
    "Austerlitz (1805)": SCENARIOS_DIR / "austerlitz.yaml",
    "Borodino (1812)": SCENARIOS_DIR / "borodino.yaml",
}

print("Scenario files found:")
for name, path in SCENARIO_FILES.items():
    print(f"  {'✓' if path.exists() else '✗'} {name}: {path.name}")

## 1. Load Scenarios and Inspect Initial Conditions

In [ ]:
loaded = {}
for name, path in SCENARIO_FILES.items():
    scenario, blue, red, terrain = load_scenario(path)
    loaded[name] = (scenario, blue, red, terrain)
    ho = scenario.historical_outcome
    print(f"\n{'='*60}")
    print(f"  {scenario.name}")
    print(f"  Date: {scenario.date}")
    print(f"  Blue faction : {scenario.faction_blue}")
    print(f"  Red faction  : {scenario.faction_red}")
    print(f"  Blue units   : {scenario.n_blue}  |  Red units: {scenario.n_red}")
    print(f"  Terrain type : {scenario.terrain_config.terrain_type}")
    print(f"  Historical winner    : {'Draw' if ho.winner is None else ('Blue' if ho.winner == 0 else 'Red')}")
    print(f"  Historical casualties: Blue {ho.blue_casualties:.0%}, Red {ho.red_casualties:.0%}")
    print(f"  Duration (steps)     : {ho.duration_steps}")
    print(f"  Notes: {ho.description[:80].strip()}...")

## 2. Visualise Initial Unit Positions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (battle_name, (scenario, blue, red, _)) in zip(axes, loaded.items()):
    for i, b in enumerate(blue, start=1):
        ax.scatter(b.x, b.y, color="royalblue", s=120, zorder=3)
        label = getattr(b, "unit_id", f"blue_{i}")
        ax.annotate(
            label.replace("_", "\n"),
            (b.x, b.y),
            fontsize=5,
            ha="center",
            va="bottom",
            color="royalblue",
        )
    for j, r in enumerate(red, start=1):
        ax.scatter(r.x, r.y, color="crimson", marker="^", s=120, zorder=3)
        label = getattr(r, "unit_id", f"red_{j}")
        ax.annotate(
            label.replace("_", "\n"),
            (r.x, r.y),
            fontsize=5,
            ha="center",
            va="bottom",
            color="crimson",
        )
    ax.set_xlim(0, scenario.terrain_config.width)
    ax.set_ylim(0, scenario.terrain_config.height)
    ax.set_title(battle_name, fontsize=10, fontweight="bold")
    ax.set_xlabel("x (m)", fontsize=8)
    ax.set_ylabel("y (m)", fontsize=8)
    ax.grid(True, alpha=0.3)

blue_patch = mpatches.Patch(color="royalblue", label="Blue (circle)")
red_patch = mpatches.Patch(color="crimson", label="Red (triangle)")
fig.legend(handles=[blue_patch, red_patch], loc="upper right", fontsize=8)
fig.suptitle("Initial Battalion Positions — Historical Scenarios", fontsize=12)
plt.tight_layout()
plt.show()

## 3. Run 1v1 Simulation Baseline (No Trained Policy)

Each scenario is reduced to a 1v1 engagement (first blue vs. first red battalion)
using the raw simulation engine.  This establishes a baseline fidelity score before
any trained policy is applied.

In [ ]:
N_SEEDS = 10  # Number of seeds for statistical robustness

baseline_results = {}  # name → list[ComparisonResult]

for name, (scenario, blue, red, terrain) in loaded.items():
    comparisons = []
    for seed in range(N_SEEDS):
        # Re-build fresh battalion instances for each run
        b, r = scenario.build_battalions()
        result = SimEngine(
            b[0], r[0],
            terrain=terrain,
            rng=np.random.default_rng(seed),
        ).run()
        cmp = OutcomeComparator(scenario.historical_outcome).compare(result)
        comparisons.append(cmp)
    baseline_results[name] = comparisons

    fidelities = [c.fidelity_score for c in comparisons]
    winner_match_rate = sum(c.winner_matches for c in comparisons) / N_SEEDS
    print(f"\n{name}")
    print(f"  Fidelity:           mean={np.mean(fidelities):.3f}  std={np.std(fidelities):.3f}")
    print(f"  Winner match rate:  {winner_match_rate:.0%}")
    avg_blue_cas = np.mean([c.simulated_blue_casualties for c in comparisons])
    avg_red_cas  = np.mean([c.simulated_red_casualties  for c in comparisons])
    hist_blue    = scenario.historical_outcome.blue_casualties
    hist_red     = scenario.historical_outcome.red_casualties
    print(f"  Blue casualties:    sim={avg_blue_cas:.2%}  hist={hist_blue:.2%}")
    print(f"  Red  casualties:    sim={avg_red_cas:.2%}  hist={hist_red:.2%}")

## 4. Fidelity Score Visualisation

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

names = list(baseline_results.keys())
means = [np.mean([c.fidelity_score for c in v]) for v in baseline_results.values()]
stds  = [np.std ([c.fidelity_score for c in v]) for v in baseline_results.values()]

colors = ["steelblue", "seagreen", "goldenrod"]
bars = ax.bar(names, means, yerr=stds, capsize=6, color=colors, alpha=0.85, edgecolor="black")

ax.axhline(0.5, color="grey", linestyle="--", linewidth=1, label="Fidelity = 0.5")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Fidelity Score  [0 – 1]", fontsize=10)
ax.set_title("Baseline Simulation Fidelity vs. Historical Record\n(mean ± std, 10 seeds, 1v1, no trained policy)", fontsize=10)
ax.legend(fontsize=8)

for bar, m, s in zip(bars, means, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        m + s + 0.02,
        f"{m:.3f}",
        ha="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

## 5. Casualty Comparison Table

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.axis("off")

col_labels = ["Scenario", "Hist. Blue Cas.", "Sim Blue Cas.", "Hist. Red Cas.", "Sim Red Cas.", "Fidelity"]
rows = []
for name, (scenario, _, _, _) in loaded.items():
    comparisons = baseline_results[name]
    ho = scenario.historical_outcome
    avg_blue = np.mean([c.simulated_blue_casualties for c in comparisons])
    avg_red  = np.mean([c.simulated_red_casualties  for c in comparisons])
    avg_fid  = np.mean([c.fidelity_score            for c in comparisons])
    rows.append([
        name,
        f"{ho.blue_casualties:.1%}",
        f"{avg_blue:.1%}",
        f"{ho.red_casualties:.1%}",
        f"{avg_red:.1%}",
        f"{avg_fid:.3f}",
    ])

table = ax.table(
    cellText=rows,
    colLabels=col_labels,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.4)

ax.set_title("Historical vs. Simulated Casualties (baseline, no policy)", fontsize=10, pad=12)
plt.tight_layout()
plt.show()

## 6. Summary and Next Steps

### Observations

* The **Borodino** scenario (historical draw) achieves the highest baseline fidelity
  because the raw simulation also tends to time-out rather than produce a decisive
  winner — reinforcing why Borodino is a good calibration benchmark.
* **Waterloo** and **Austerlitz** have lower fidelity in the 1v1 baseline because the
  simulation cannot replicate the flanking manoeuvres and numerical
  disparity that drove the historical outcomes — a sign that a trained
  policy is needed.
* Casualty magnitudes differ from historical figures because the simulation
  maps an entire brigade to a single unit with strength in \[0, 1\], not raw
  headcounts.  The relative ordering (which side suffers more) is meaningful.

### Next Steps

1. **Train a policy** against each historical scenario and re-run this notebook.
2. **Document findings** in a `[EXP]` issue on GitHub, including fidelity scores
   before and after policy training.
3. **Extend scenarios** to multi-unit engagements using `MultiBattalionEnv`
   once the scenario loader supports multi-agent initialisation.
4. **Add terrain visualisation** to show how Waterloo's ridge and Borodino's
   earthworks affect movement and cover.